# Step 06. RRT (Rapidly-exploring Random Tree)

> **출처**: PythonRobotics (https://github.com/AtsushiSakai/PythonRobotics)  
> Original author: Atsushi Sakai (@Atsushi_twi) / License: MIT  
>  
> **교육 주관**: Hanyang University, Automotive Intelligence Laboratory (AILAB)

격자 기반(A\*)과 Frenet 기반 Local Planner(Step 04~05)를 학습했습니다.  
하지만 **좁은 통로, 고차원 공간, 비구조적 환경**에서는 격자 이산화가 한계를 가집니다.  
**RRT**는 연속 공간에서 **무작위 샘플링**으로 직접 탐색하는 샘플링 기반 계획법입니다.

## 학습 목표

1. 샘플링 기반 계획과 격자 기반 계획의 차이 이해
2. RRT 핵심 연산 (sample → nearest → steer → collision check) 구현
3. goal bias가 탐색 수렴에 미치는 영향 이해
4. 트리 성장 과정 애니메이션으로 무작위 탐색 시각화

In [ ]:
%matplotlib inline
import sys, os, math, random, time
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

from utils.grid_map import make_simple_map

ox, oy, sx, sy, gx, gy = make_simple_map()

print('환경 로드 완료  (Day 1과 동일한 맵에서 RRT vs A* 비교 가능)')
print(f'  탐색 공간: ({min(ox):.0f}~{max(ox):.0f}, {min(oy):.0f}~{max(oy):.0f})')
print(f'  출발: ({sx}, {sy})  ->  목표: ({gx}, {gy})')


## 1. 샘플링 기반 계획이란?

### 이론 배경

| 구분 | 격자 기반 (A\*) | 샘플링 기반 (RRT) |
|------|---------------|------------------|
| 탐색 방식 | 격자 전체 시스템적 탐색 | 무작위 샘플 기반 탐색 |
| 연속 공간 | 이산화 필요 (해상도 제약) | 자연스럽게 표현 가능 |
| 최적성 | 보장 (해상도 내) | 미보장 (확률적 완전성) |
| 좁은 통로 | 해상도 낮으면 어려움 | 비교적 쉬움 |

**RRT 핵심 아이디어**:

```
① 공간에서 무작위 점 q_rand 샘플
② 트리에서 q_rand 에 가장 가까운 노드 q_near 탐색
③ q_near 에서 q_rand 방향으로 step_size 이동 -> q_new
④ q_near -> q_new 경로 충돌 없으면 트리에 추가
⑤ q_new 가 목표 반경 내이면 종료
```

**확률적 완전성**: 반복 횟수 → ∞ 이면 경로가 존재할 때 반드시 찾음

In [ ]:
class RRTNode:
    # RRT 트리의 노드 — parent 포인터로 역추적
    def __init__(self, x, y):
        self.x      = x
        self.y      = y
        self.parent = None  # 경로 역추적용
        self.cost   = 0.0   # 시작점으로부터 거리

    def __repr__(self):
        return f'RRTNode(x={self.x:.1f}, y={self.y:.1f}, cost={self.cost:.2f})'

n1 = RRTNode(0.0, 0.0)
n2 = RRTNode(3.0, 4.0)
n2.parent = n1
n2.cost   = math.hypot(3.0, 4.0)

print('RRTNode 예시')
print(f'  루트: {n1}')
print(f'  자식: {n2}  (parent -> 루트)')
print()
print('  비교: Dijkstra/A* Node 의 parent_index = 정수 dict 키')
print('        RRTNode.parent = 노드 객체 직접 참조 (연속 공간이므로 1D 키 불필요)')


## 2. RRT 핵심 함수

### 이론 배경

각 함수의 역할:

- **sample_random**: `goal_bias=0.1`로 10% 확률로 목표 직접 샘플 → 수렴 가속
- **nearest_node**: O(N) 순차 탐색. KD-Tree 사용 시 O(log N)으로 개선 가능
- **steer**: step_size로 트리 밀도 제어. 작을수록 정밀하지만 반복 수 증가
- **is_collision_free_rrt**: 선분을 작은 간격으로 분할 후 각 점 검사

In [ ]:
STEP_SIZE   = 5.0
GOAL_RADIUS = 5.0
MAX_ITER    = 3000

def sample_random(gx, gy, area, goal_bias=0.1):
    # goal_bias 확률로 목표점을 직접 샘플 -> 수렴 가속
    if random.random() < goal_bias:
        return gx, gy
    return random.uniform(area[0], area[1]), random.uniform(area[2], area[3])

def nearest_node(tree, rx, ry):
    # 트리에서 (rx,ry)에 가장 가까운 노드 O(N)
    return min(tree, key=lambda n: math.hypot(n.x-rx, n.y-ry))

print(f'STEP_SIZE={STEP_SIZE}m,  GOAL_RADIUS={GOAL_RADIUS}m')
print('sample_random : 무작위 좌표 (10% 확률로 목표 직접)')
print('nearest_node  : 트리에서 가장 가까운 노드')


핵심 함수 두 번째 묶음 — **steer**(전진)와 **충돌 검사**:

In [ ]:
def steer(from_node, rx, ry, step_size=STEP_SIZE):
    # from_node 에서 (rx,ry) 방향으로 step_size 만큼 이동한 새 노드 생성
    theta = math.atan2(ry - from_node.y, rx - from_node.x)
    nn = RRTNode(from_node.x + step_size*math.cos(theta),
                 from_node.y + step_size*math.sin(theta))
    nn.parent = from_node; nn.cost = from_node.cost + step_size
    return nn

def is_collision_free_rrt(from_node, to_node, ox, oy, radius=1.5):
    # 두 노드를 잇는 선분을 잘게 나눠 각 점이 장애물에서 radius 이상인지 검사
    dx, dy = to_node.x - from_node.x, to_node.y - from_node.y
    steps = int(math.hypot(dx, dy) / radius) + 1
    for i in range(steps):
        t = i / max(steps - 1, 1)
        px, py = from_node.x + t*dx, from_node.y + t*dy
        if any(math.hypot(iox-px, ioy-py) <= radius for iox, ioy in zip(ox, oy)):
            return False
    return True

print('steer                : 방향으로 step_size 이동 후 새 노드')
print('is_collision_free_rrt: 선분-장애물 충돌 검사')


In [ ]:
# RRT 한 번의 확장: sample -> nearest -> steer (Dijkstra의 "한 번의 확장"에 해당)
random.seed(7)
demo_area = [min(ox)+1, max(ox)-1, min(oy)+1, max(oy)-1]
demo_tree = [RRTNode(sx, sy)]
for _ in range(8):                       # 데모용 작은 트리
    rx_, ry_ = sample_random(gx, gy, demo_area, goal_bias=0.0)
    near = nearest_node(demo_tree, rx_, ry_); nn = steer(near, rx_, ry_)
    if is_collision_free_rrt(near, nn, ox, oy): demo_tree.append(nn)

q_rand = (gx-10, gy-25)                   # 강조할 한 번의 연산
q_near = nearest_node(demo_tree, *q_rand); q_new = steer(q_near, *q_rand)

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(ox, oy, '.k', ms=2, alpha=0.4)
ax.plot(sx, sy, 'og', ms=12, label='Start'); ax.plot(gx, gy, 'xb', ms=12, mew=2.5, label='Goal')
for n in demo_tree:
    if n.parent: ax.plot([n.x, n.parent.x], [n.y, n.parent.y], '-', color='cyan', lw=1, alpha=0.7)
ax.plot(*q_rand, '*', color='orange', ms=20, zorder=5, label='1. q_rand (sample)')
ax.plot(q_near.x, q_near.y, 'o', color='red', ms=12, mfc='none', mew=2.5, label='2. q_near (nearest)')
ax.plot([q_near.x, q_rand[0]], [q_near.y, q_rand[1]], ':', color='gray', lw=1.5)
ax.annotate('', xy=(q_new.x, q_new.y), xytext=(q_near.x, q_near.y),
            arrowprops=dict(arrowstyle='-|>', color='red', lw=2.5, mutation_scale=20))
ax.plot(q_new.x, q_new.y, 's', color='red', ms=11, zorder=5, label='3. q_new (steer)')
ax.set_aspect('equal'); ax.grid(True, alpha=0.3); ax.legend(loc='upper left', fontsize=8)
ax.set_title('One RRT Iteration: sample -> nearest -> steer')
plt.tight_layout(); plt.show()
print('q_new가 충돌 없으면 트리에 추가. 전체 탐색 = 이 연산을 목표 도달까지 반복')


## 3. RRT 탐색

### 이론 배경

탐색 종료 조건: 새 노드가 `GOAL_RADIUS` 이내에 생성될 때.  
정확한 목표점 도달을 요구하면 수렴하지 못할 수 있으므로 반경으로 완화합니다.

경로 역추적: `parent` 포인터를 따라 목표 → 출발로 역추적 후 뒤집습니다.

In [ ]:
random.seed(42)   # 재현성 고정
area = [min(ox)+1, max(ox)-1, min(oy)+1, max(oy)-1]
tree = [RRTNode(sx, sy)]
path_found = None

t0 = time.time()
for i in range(MAX_ITER):
    rx, ry = sample_random(gx, gy, area)          # ① 샘플
    nearest = nearest_node(tree, rx, ry)          # ② 가장 가까운 노드
    new_nd = steer(nearest, rx, ry)               # ③ step_size 이동
    if not is_collision_free_rrt(nearest, new_nd, ox, oy):
        continue                                   # 충돌이면 버림
    tree.append(new_nd)
    if math.hypot(new_nd.x - gx, new_nd.y - gy) < GOAL_RADIUS:   # 목표 반경 도달
        goal_node = steer(new_nd, gx, gy, step_size=math.hypot(new_nd.x-gx, new_nd.y-gy))
        goal_node.parent = new_nd; tree.append(goal_node); path_found = goal_node
        break
t1 = time.time()

path_nodes = []                                    # parent 체인 역추적
node = path_found
while node: path_nodes.append(node); node = node.parent
path_nodes.reverse()
path_x = [n.x for n in path_nodes]; path_y = [n.y for n in path_nodes]

print(f'탐색 완료 ({(t1-t0)*1000:.0f} ms),  반복 {i+1:,}회')
print(f'트리 {len(tree):,}개 노드,  경로 {len(path_nodes)}개 점,  길이 {path_nodes[-1].cost:.1f} m')


## 4. 결과 시각화

### 이론 배경

- **왼쪽**: 트리 전체 (cyan 브랜치) + 경로 (빨강) — 무작위 탐색이 공간을 커버하는 모습
- **오른쪽**: 발견된 경로 단독 — A\* 경로와 비교 가능 (A\*가 최단, RRT는 근사)

RRT는 `seed`를 고정하지 않으면 실행마다 다른 경로가 나옵니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax in axes:
    ax.plot(ox, oy, '.k', ms=2)
    ax.plot(sx, sy, 'og', ms=10, label='Start')
    ax.plot(gx, gy, 'xb', ms=10, mew=2, label='Goal')
    ax.set_aspect('equal'); ax.grid(True, alpha=0.3)

# Left: 전체 트리 + 경로
for node in tree:
    if node.parent:
        axes[0].plot([node.x, node.parent.x], [node.y, node.parent.y],
                     '-', color='cyan', lw=0.5, alpha=0.5)
axes[0].plot(path_x, path_y, '-r', lw=2, label='Path')
axes[0].set_title(f'RRT Tree ({len(tree):,} nodes)')
axes[0].legend(fontsize=8)

# Right: 경로만
axes[1].plot(path_x, path_y, '-r', lw=2.5,
             label=f'RRT Path ({path_nodes[-1].cost:.0f} m)')
axes[1].set_title('RRT Found Path')
axes[1].legend(fontsize=8)

plt.suptitle('RRT — Sampling-based Path Planning', fontsize=13)
plt.tight_layout()
plt.show()
print('RRT는 랜덤성 때문에 실행마다 다른 경로가 나옴 (seed 고정으로 재현 가능)')


## 5. 트리 성장 애니메이션

### 이론 배경

트리가 성장하는 과정에서 확인할 포인트:

1. **무작위성**: 처음에는 불규칙하게 퍼짐
2. **목표 편향(goal bias)**: 10% 확률로 목표를 향해 직접 뻗어나가는 브랜치
3. **장애물 근처**: 충돌 노드는 추가되지 않아 빈 공간이 생김
4. **수렴**: 목표 반경에 노드가 도달하면 빨간 경로 표시

In [ ]:
# 트리 성장 애니메이션 (50노드마다 1프레임)
BATCH = 50
n_frames = math.ceil(len(tree) / BATCH) + 5

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(ox, oy, '.k', ms=2, alpha=0.5, zorder=1)
ax.plot(sx, sy, 'og', ms=12, zorder=5, label='Start')
ax.plot(gx, gy, 'xb', ms=12, mew=2.5, zorder=5, label='Goal')
ax.set_aspect('equal'); ax.grid(True, alpha=0.2); ax.legend(loc='upper left')
ax.set_title('RRT: Tree Growing Animation')

edge_lines = []  # 누적 엣지
path_line, = ax.plot([], [], '-r', lw=3, zorder=4)
title_text = ax.set_title('RRT: exploring...')

def update_rrt(frame):
    if frame < n_frames - 5:
        start_k = frame * BATCH
        end_k   = min(start_k + BATCH, len(tree))
        for k in range(start_k, end_k):
            node = tree[k]
            if node.parent:
                ln, = ax.plot([node.x, node.parent.x],
                              [node.y, node.parent.y],
                              '-', color='cyan', lw=0.6, alpha=0.6, zorder=2)
                edge_lines.append(ln)
        title_text.set_text(f'RRT exploring... {min(end_k, len(tree)):,}/{len(tree):,} nodes')
        path_line.set_data([], [])
    else:
        title_text.set_text(
            f'Done!  Tree: {len(tree):,} nodes  |  Path: {path_nodes[-1].cost:.0f} m')
        path_line.set_data(path_x, path_y)
    return edge_lines + [path_line, title_text]

ani_rrt = FuncAnimation(fig, update_rrt,
                        frames=n_frames, interval=120,
                        repeat=False, blit=False)
HTML(ani_rrt.to_jshtml())


## 6. Day 1 + Day 2 알고리즘 종합 비교

### 이론 배경

실습에서 학습한 5가지 알고리즘의 특성 비교.  
어떤 환경에서 어떤 알고리즘을 선택할지 판단하는 기준이 됩니다.

- **Dijkstra / A\***: 최적성 보장, 구조화된 도로 탐색
- **Hybrid A\***: 차량 기구학 반영, 주차/좁은 공간
- **Frenet LP**: 차선 유지/변경 효율적 표현
- **RRT**: 비구조적/고차원 환경, 최적성 불필요할 때

In [ ]:
print('Day 1 + Day 2 알고리즘 종합 비교')
print('=' * 72)
print(f'  {"알고리즘":<14}  {"상태 공간":<16}  {"이동 모델":<14}  {"최적성":<8}  적합 환경')
print('  ' + '-' * 68)
rows = [
    ('Dijkstra', '격자 (ix,iy)', '8방향',      '보장',   '구조화된 도로'),
    ('A*',       '격자 (ix,iy)', '8방향+h(n)', '보장',   '구조화된 도로'),
    ('Hybrid A*','연속 (x,y,θ)','Bicycle',    '미보장', '주차/저속 정밀'),
    ('Frenet LP','Frenet (s,d)', 'd 오프셋',   '미보장', '차선 유지/변경'),
    ('Frenet Opt','Frenet (s,d,t)','Quintic 궤적','미보장', '동적 회피/재계획'),
    ('RRT',      '연속 (x,y)',   '랜덤 샘플',  '미보장', '좁은 통로/복잡'),
]
for algo, space, model, opt, env in rows:
    print(f'  {algo:<14}  {space:<16}  {model:<14}  {opt:<8}  {env}')
print('=' * 72)
print()
print('Day 1 (Global Planning) + Day 2 (Local Planning) 실습 완료!')
